In [1]:
!pip install -q faiss-cpu sentence-transformers pypdf

import os
import re
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

# ============================
# 1) تحميل وتنظيف ملفات PDF
# ============================
extract_path = '/kaggle/input/datasets/abedgaha/dataforrag/LIMITSFilesForRAG'

def clean_text(t):
    t = re.sub(r'\s+', ' ', t)
    t = re.sub(r'[^\x00-\x7f]', '', t)
    return t.strip()

documents = []
for filename in os.listdir(extract_path):
    if filename.endswith(".pdf"):
        reader = PdfReader(os.path.join(extract_path, filename))
        for page in reader.pages:
            text = clean_text(page.extract_text())
            documents.append(text)

print("Loaded:", len(documents), "pages")

# ============================
# 2) تقطيع النصوص يدويًا
# ============================
def split_text(text, chunk_size=600, overlap=120):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = []
for doc in documents:
    chunks.extend(split_text(doc))

print("Total chunks:", len(chunks))

# ============================
# 3) Embeddings
# ============================
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(chunks, convert_to_numpy=True)

# ============================
# 4) بناء FAISS Index
# ============================
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# ============================
# 5) حفظ FAISS + النصوص
# ============================
save_dir = "/kaggle/working/medical_faiss_db"
os.makedirs(save_dir, exist_ok=True)

faiss.write_index(index, f"{save_dir}/index.faiss")
np.save(f"{save_dir}/texts.npy", np.array(chunks, dtype=object))

print("FAISS DB saved to:", save_dir)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.0 MB/s eta 0:00:00
Loaded: 103 pages
Total chunks: 801


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS DB saved to: /kaggle/working/medical_faiss_db
